In [59]:
df.isnull().sum()

PassengerId       0
Survived          0
Pclass            0
Name              0
Sex               0
Age             177
SibSp             0
Parch             0
Ticket            0
Fare              0
Cabin           687
Embarked          0
Embarked_num      0
Sex_num           0
Family            0
Fare_cat          0
Age_fix           0
dtype: int64

In [3]:
df=pd.read_csv('C:/Users/Danya/Downloads/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [36]:
#смена порта посадки со строкового на чисовое значение
df['Embarked'] = df['Embarked'].fillna('S')
df['Embarked_num'] = np.where(df['Embarked'] == 'S', 0,
                               np.where(df['Embarked'] == 'C', 1, 2))
df['Embarked_num'].value_counts()

Embarked_num
0    646
1    168
2     77
Name: count, dtype: int64

In [41]:
#аналогично для пола пассажиров
df['Sex_num']=np.where(df['Sex']== 'male', 0, 1)
df['Sex_num'].value_counts()

Sex_num
0    577
1    314
Name: count, dtype: int64

In [6]:
#создаем столбец для опередения пассажиров-одиночек и семейных
df['Family']=df['Parch']+df['SibSp']
df['Family'].value_counts()

Family
0     537
1     161
2     102
3      29
5      22
4      15
6      12
10      7
7       6
Name: count, dtype: int64

In [58]:
#столбец с категоризацией билетов по цене
df['Fare_cat']=pd.cut(df['Fare'], bins=[-1, 7.91, 14.45, 31, 513],
                      labels=['cheap', 'medium', 'expensive', 'elite'])
df['Fare_cat'].value_counts().sum()

np.int64(891)

In [46]:
#вручную заполненный столбец возраста, пропуски заполнены средним
df['Age_fix']=df['Age'].fillna(df['Age'].mean())

In [76]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Данные
X = df[['Age', 'Family', 'Sex_num', 'Embarked_num', 'Pclass', 'Fare_cat']]
y = df['Survived']

#Разбиение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=777, stratify=y
)

#числовие и категориальные признаки
num_features = ['Age', 'Family', 'Sex_num', 'Embarked_num']
cat_features = ['Fare_cat']

#предобработка
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_features)
])

#модели обучения
models = {
    'Logistic Regression': LogisticRegression(random_state=777),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=777),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=10, random_state=777),
    'SVM': SVC(random_state=777, probability=True),  # важно: probability=True для predict_proba
    'KNN': KNeighborsClassifier(n_neighbors=100),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=777, use_label_encoder=False, eval_metric='logloss')
}

#обучение и оценка
results = {}

for name, model in models.items():
       
    # Создаём пайплайн: предобработка + модель
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Кросс-валидация на train
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
    
    # Обучение на всём train
    pipeline.fit(X_train, y_train)
    
    # Предсказания на тесте
    y_pred = pipeline.predict(X_test)
    
    # Вероятности для ROC-AUC (если модель поддерживает)
    if hasattr(pipeline, 'predict_proba'):
        y_prob = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_prob = None
    
    # Метрики
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan
    
    # Сохранение результатов
    results[name] = {
        'CV_Accuracy': np.mean(cv_scores),
        'Test_Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'ROC_AUC': roc_auc
    }

#печать результатов
results_df = pd.DataFrame(results).T
print("Results:")
print(results_df.round(4))


c:\Users\Danya\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:199: UserWarning: [07:03:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Danya\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:199: UserWarning: [07:03:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Danya\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:199: UserWarning: [07:03:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Danya\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:199: UserWarning: [07:03:20] WARNING: C:\actio

Results:
                     CV_Accuracy  Test_Accuracy  Precision  Recall      F1  \
Logistic Regression       0.7886         0.8134     0.7955  0.6863  0.7368   
Random Forest             0.7674         0.8060     0.8049  0.6471  0.7174   
Gradient Boosting         0.8110         0.8507     0.8605  0.7255  0.7872   
SVM                       0.8058         0.8358     0.8222  0.7255  0.7708   
KNN                       0.7873         0.8134     0.7955  0.6863  0.7368   
XGBoost                   0.7753         0.7985     0.7857  0.6471  0.7097   

                     ROC_AUC  
Logistic Regression   0.7668  
Random Forest         0.7987  
Gradient Boosting     0.8129  
SVM                   0.8242  
KNN                   0.8084  
XGBoost               0.7987  
